# Effectiveness - baselines vs MetaSpace

Ce notebook reprend uniquement les resultats deja sauvegardes. Il ne relance pas les modeles.

Objectif: comparer toutes les baselines disponibles avec MetaSpace sur les 551 paires de tables, puis afficher le tableau MetaSpace avec tous les classifiers disponibles.

Note importante: `F1@GT` force un top-k par paire de tables, avec `k = nombre reel de matchs`. Si une methode a tous ses scores a zero, le top-k depend de l'ordre des lignes: le score est alors un baseline arbitraire, pas une vraie methode informative.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/Users/nahawandkired/Documents/metamatch')
EFFECTIVENESS_DIR = ROOT / 'outputs/exp_occidata/reports/meeting_baselines_vs_metamatch/effectiveness'

summary_path = EFFECTIVENESS_DIR / 'effectiveness_ALL_AVAILABLE_methods_f1GT_summary_551pairs.csv'
pair_path = EFFECTIVENESS_DIR / 'effectiveness_ALL_AVAILABLE_methods_f1GT_by_pair_551pairs.csv'
article_path = EFFECTIVENESS_DIR / 'effectiveness_ARTICLE_all_baselines_plus_metaspace_classifiers_f1GT.csv'
coverage_path = EFFECTIVENESS_DIR / 'effectiveness_ALL_AVAILABLE_methods_score_coverage_by_fold.csv'
classifiers_path = EFFECTIVENESS_DIR / 'metaspace_all_classifiers_61features_summary.csv'
classifiers_pair_path = EFFECTIVENESS_DIR / 'metaspace_all_classifiers_61features_by_pair.csv'

summary = pd.read_csv(summary_path)
pair_df = pd.read_csv(pair_path)
article_summary = pd.read_csv(article_path)
coverage = pd.read_csv(coverage_path)
classifiers = pd.read_csv(classifiers_path)
classifiers_pair = pd.read_csv(classifiers_pair_path)

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)

## 1. Tableau effectiveness global

Metric principale: `F1@GT`, moyennee sur les 551 `pair_id`. Pour chaque paire de tables, on garde autant de predictions positives que le nombre reel de correspondances.

Le tableau inclut maintenant toutes les methodes qui ont des `predictions.parquet` dans `outputs/exp_occidata/results`.

In [ ]:
display_cols = [
    'method_label', 'mean_f1_ground_size', 'std_f1_ground_size',
    'mean_precision_ground_size', 'mean_recall_ground_size',
    'n_pair_id', 'n_folds', 'total_score_nonzero', 'max_score_max', 'source'
]
global_table = summary[display_cols].copy()
for col in ['mean_f1_ground_size','std_f1_ground_size','mean_precision_ground_size','mean_recall_ground_size','max_score_max']:
    global_table[col] = global_table[col].round(4)
display(global_table)

## 1b. Controle des scores nuls

Les lignes avec `total_score_nonzero = 0` doivent etre traitees avec prudence. Elles n'ont pas vraiment classe les candidats; le `F1@GT` obtenu vient seulement de l'ordre initial des lignes apres tri avec scores egaux.

In [ ]:
zero_score_methods = summary[summary['total_score_nonzero'].eq(0)][[
    'method', 'method_label', 'mean_f1_ground_size', 'n_pair_id', 'n_folds', 'total_score_nonzero'
]]
display(zero_score_methods)

## 2. Histogramme / barplot F1@GT moyen par methode

In [ ]:
plot_df = summary.sort_values('mean_f1_ground_size', ascending=True).copy()
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2f855a' if 'MetaSpace' in name else '#4a5568' for name in plot_df['method_label']]
ax.barh(plot_df['method_label'], plot_df['mean_f1_ground_size'], color=colors)
ax.set_xlabel('Mean F1@GT sur 551 pair_id')
ax.set_title('Effectiveness - baselines vs MetaSpace')
ax.set_xlim(0, 1)
for i, value in enumerate(plot_df['mean_f1_ground_size']):
    ax.text(value + 0.01, i, f'{value:.3f}', va='center')
plt.tight_layout()
plt.show()

## 3. Distribution par dataset/pair_id

In [ ]:
keep = summary.head(8)['method'].tolist()
dist_df = pair_df[pair_df['method'].isin(keep)].copy()
labels = summary.set_index('method')['method_label'].to_dict()
dist_df['method_label'] = dist_df['method'].map(labels).fillna(dist_df['method_label'])

fig, ax = plt.subplots(figsize=(11, 5))
dist_df.boxplot(column='f1_ground_size', by='method_label', ax=ax, rot=35, grid=False)
ax.set_title('Distribution F1@GT par pair_id')
ax.set_xlabel('')
ax.set_ylabel('F1@GT')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. MetaSpace avec tous les classifiers disponibles

Ici le tableau vient du run `metaspace_61_classifiers_notebook`: features `syn + cls + tda`, sans overlap/spectral/NLP selon la config que tu avais demandee.

In [ ]:
clf_cols = [
    'classifier', 'method_label', 'mean_f1_ground_size', 'std_f1_ground_size',
    'mean_f1_opt_train', 'std_f1_opt_train',
    'mean_precision_opt_train', 'mean_recall_opt_train', 'n_pair_id', 'feature_set'
]
clf_table = classifiers[clf_cols].sort_values('mean_f1_ground_size', ascending=False).copy()
for col in clf_table.select_dtypes(include='number').columns:
    if col != 'n_pair_id':
        clf_table[col] = clf_table[col].round(4)
display(clf_table)

In [ ]:
plot_clf = classifiers.sort_values('mean_f1_ground_size', ascending=True).copy()
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(plot_clf['method_label'], plot_clf['mean_f1_ground_size'], color='#2b6cb0')
ax.set_xlabel('Mean F1@GT sur 551 pair_id')
ax.set_title('MetaSpace - comparaison des classifiers')
ax.set_xlim(0, 1)
for i, value in enumerate(plot_clf['mean_f1_ground_size']):
    ax.text(value + 0.01, i, f'{value:.3f}', va='center')
plt.tight_layout()
plt.show()

## 5. Fichiers produits

In [ ]:
for path in [summary_path, pair_path, article_path, coverage_path, classifiers_path, classifiers_pair_path]:
    print(path)